## Segmentacja naczyń - architektura U-Net

In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu
%pip install tqdm

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 512  # obrazy będą resize'owane do IMG_SIZE x IMG_SIZE
print(f"Device: {DEVICE}")

## Wczytanie danych

In [ ]:
images, masks = [], []
for i in range(1, 16):
    fname = f"0{i}_h" if i < 10 else f"{i}_h"
    img = cv.imread(f"dataset/healthy/{fname}.jpg")
    img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
    mask = cv.imread(f"dataset/healthy_manualsegm/{fname}.tif", cv.IMREAD_GRAYSCALE)
    images.append(img)
    masks.append(mask)

print(f"Załadowano {len(images)} obrazów, rozmiar: {images[0].shape}")

## Dataset

In [ ]:
class RetinaDataset(Dataset):
    def __init__(self, images, masks, img_size=512):
        self.img_size = img_size
        self.images = []
        self.masks = []
        for img, mask in zip(images, masks):
            img_r = cv.resize(img, (img_size, img_size))
            mask_r = cv.resize(mask, (img_size, img_size), interpolation=cv.INTER_NEAREST)
            # obraz: (H, W, 3) -> (3, H, W), float32 w [0, 1]
            img_t = torch.from_numpy(img_r.transpose(2, 0, 1)).float() / 255.0
            # maska: (H, W) -> (1, H, W), binarna float32
            mask_t = torch.from_numpy((mask_r > 0).astype(np.float32)).unsqueeze(0)
            self.images.append(img_t)
            self.masks.append(mask_t)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx], self.masks[idx]


# podział train/test: 12 treningowe, 3 testowe
train_ds = RetinaDataset(images[:12], masks[:12], IMG_SIZE)
test_ds  = RetinaDataset(images[12:], masks[12:], IMG_SIZE)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False)

print(f"Treningowych: {len(train_ds)}, testowych: {len(test_ds)}")

## Architektura U-Net

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()

        # encoder
        self.enc = nn.ModuleList()
        self.pool = nn.MaxPool2d(2)
        ch = in_channels
        for f in features:
            self.enc.append(DoubleConv(ch, f))
            ch = f

        # bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        # decoder
        self.dec_up   = nn.ModuleList()
        self.dec_conv = nn.ModuleList()
        rev = list(reversed(features))
        ch = features[-1] * 2
        for f in rev:
            self.dec_up.append(nn.ConvTranspose2d(ch, f, 2, stride=2))
            self.dec_conv.append(DoubleConv(f * 2, f))
            ch = f

        self.final = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        skips = []
        for layer in self.enc:
            x = layer(x)
            skips.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        skips = skips[::-1]
        for i, (up, conv) in enumerate(zip(self.dec_up, self.dec_conv)):
            x = up(x)
            x = torch.cat([skips[i], x], dim=1)
            x = conv(x)

        return self.final(x)


model = UNet().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parametrów modelu: {total_params:,}")

## Trening

In [ ]:
# BCEWithLogitsLoss - numerycznie stabilne połączenie sigmoid + BCE
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 20
train_losses = []

epoch_bar = tqdm(range(EPOCHS), desc="Trening", unit="epoch")
for epoch in epoch_bar:
    model.train()
    epoch_loss = 0.0
    batch_bar = tqdm(train_loader, desc=f"Epoch {epoch+1:2d}/{EPOCHS}", leave=False, unit="batch")
    for imgs, msks in batch_bar:
        imgs, msks = imgs.to(DEVICE), msks.to(DEVICE)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, msks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        batch_bar.set_postfix(loss=f"{loss.item():.4f}")
    avg = epoch_loss / len(train_loader)
    train_losses.append(avg)
    epoch_bar.set_postfix(avg_loss=f"{avg:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(train_losses)
plt.xlabel("Epoch")
plt.ylabel("BCE Loss")
plt.title("Krzywa strat treningowych")
plt.tight_layout()
plt.show()

## Ewaluacja

In [ ]:
def metrics(pred_bin, gt_bin):
    pred = pred_bin.astype(bool)
    gt   = gt_bin.astype(bool)
    tp = np.logical_and(pred, gt).sum()
    tn = np.logical_and(~pred, ~gt).sum()
    fp = np.logical_and(pred, ~gt).sum()
    fn = np.logical_and(~pred, gt).sum()
    accuracy    = (tp + tn) / (tp + tn + fp + fn + 1e-8)
    sensitivity = tp / (tp + fn + 1e-8)
    specificity = tn / (tn + fp + 1e-8)
    bal_a = 0.5 * (sensitivity + specificity)
    bal_g = np.sqrt(sensitivity * specificity)
    return accuracy, sensitivity, specificity, bal_a, bal_g


model.eval()
rows = []
with torch.no_grad():
    for i, (img_t, mask_t) in enumerate(test_loader):
        logits = model(img_t.to(DEVICE))
        prob   = torch.sigmoid(logits).cpu().squeeze().numpy()
        pred_bin = (prob > 0.5).astype(np.uint8)
        gt_bin   = mask_t.squeeze().numpy().astype(np.uint8)
        acc, sens, spec, ba, bg = metrics(pred_bin, gt_bin)
        rows.append((i + 13, acc, sens, spec, ba, bg))

print("idx\tAcc\tSens\tSpec\tBal_A\tBal_G")
for idx, acc, sens, spec, ba, bg in rows:
    print(f"{idx}\t{acc:.3f}\t{sens:.3f}\t{spec:.3f}\t{ba:.3f}\t{bg:.3f}")

avg = np.mean(np.array([r[1:] for r in rows]), axis=0)
print("avg\t" + "\t".join(f"{v:.3f}" for v in avg))

## Wizualizacja

In [ ]:
model.eval()
fig, axes = plt.subplots(len(test_ds), 3, figsize=(12, 4 * len(test_ds)), constrained_layout=True)

with torch.no_grad():
    for i, (img_t, mask_t) in enumerate(test_loader):
        logits   = model(img_t.to(DEVICE))
        prob     = torch.sigmoid(logits).cpu().squeeze().numpy()
        pred_bin = (prob > 0.5).astype(np.uint8)
        gt_bin   = mask_t.squeeze().numpy().astype(np.uint8)

        img_vis = img_t.squeeze().permute(1, 2, 0).numpy()

        axes[i, 0].imshow(img_vis)
        axes[i, 0].set_title(f"Obraz {i+13}")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(gt_bin, cmap="gray")
        axes[i, 1].set_title("Ground truth")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(pred_bin, cmap="gray")
        axes[i, 2].set_title("Predykcja U-Net")
        axes[i, 2].axis("off")

plt.show()